# 🎓 Student Performance Prediction
**Begginer-Level ML Project** — Linear Regression · Feature Engineering · Label Encoding

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('dark_background')
%matplotlib inline

## 1️⃣ Load Data

In [ ]:
df = pd.read_csv('../data/students.csv')
print(f'Shape: {df.shape}')
df.head()

## 2️⃣ EDA

In [ ]:
df.describe().T

In [ ]:
print('Missing values:\n', df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['exam_score'].hist(bins=30, ax=axes[0], color='#00d4ff'); axes[0].set_title('Score Distribution')
df.groupby('test_preparation_course')['exam_score'].mean().plot.bar(ax=axes[1], color=['#f87171','#34d399']); axes[1].set_title('Test Prep vs Avg Score')
df.plot.scatter('study_hours_per_day','exam_score', ax=axes[2], alpha=0.3, color='#a78bfa'); axes[2].set_title('Study Hours vs Score')
plt.tight_layout()

## 3️⃣ Data Cleaning

In [ ]:
df_clean = df.copy()

# Fill numerics with median, categoricals with mode
for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# IQR outlier capping on exam_score
Q1, Q3 = df_clean['exam_score'].quantile([0.25, 0.75])
IQR    = Q3 - Q1
df_clean['exam_score'] = df_clean['exam_score'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print('After cleaning:', df_clean.isnull().sum().sum(), 'missing values')
df_clean.shape

## 4️⃣ Feature Engineering

In [ ]:
df_feat = df_clean.copy()

df_feat['study_sleep_ratio'] = df_feat['study_hours_per_day'] / (df_feat['sleep_hours'] + 1e-6)
df_feat['attendance_rate']   = 1 - df_feat['absences'] / (df_feat['absences'].max() + 1)
df_feat['support_score']     = (
    (df_feat['lunch'] == 'standard').astype(int) +
    (df_feat['test_preparation_course'] == 'completed').astype(int) +
    (df_feat['extracurricular_activities'] == 'Yes').astype(int)
)

edu_map = {'some high school':0,'high school':1,'some college':2,
           "associate's degree":3,"bachelor's degree":4,"master's degree":5}
df_feat['parental_edu_level'] = df_feat['parental_level_of_education'].map(edu_map)

df_feat['study_x_prep'] = (
    df_feat['study_hours_per_day'] *
    (df_feat['test_preparation_course'] == 'completed').astype(int)
)

print('New features added: study_sleep_ratio, attendance_rate, support_score, parental_edu_level, study_x_prep')
df_feat[['study_sleep_ratio','attendance_rate','support_score','parental_edu_level','study_x_prep']].head()

## 5️⃣ Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df_feat.copy()

binary_map = {
    'gender':                    {'Male':0,'Female':1},
    'lunch':                     {'standard':1,'free/reduced':0},
    'test_preparation_course':   {'none':0,'completed':1},
    'extracurricular_activities':{'No':0,'Yes':1},
}
for col, mapping in binary_map.items():
    df_enc[col] = df_enc[col].map(mapping)

le = LabelEncoder()
df_enc['race_ethnicity'] = le.fit_transform(df_enc['race_ethnicity'].astype(str))

df_enc.drop(columns=['parental_level_of_education'], inplace=True)
df_enc.head()

## 6️⃣ Train Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

FEATURES = ['gender','race_ethnicity','lunch','test_preparation_course',
            'study_hours_per_day','absences','sleep_hours',
            'extracurricular_activities','parental_edu_level',
            'study_sleep_ratio','attendance_rate','support_score','study_x_prep']

X = df_enc[FEATURES]
y = df_enc['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LinearRegression()),
])
pipe.fit(X_train, y_train)

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
print(f'CV R² scores: {cv_scores.round(4)}')
print(f'Mean CV R²  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 7️⃣ Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = pipe.predict(X_test)

print(f'MAE  : {mean_absolute_error(y_test, y_pred):.3f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}')
print(f'R²   : {r2_score(y_test, y_pred):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, s=15, color='#00d4ff')
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn,mx],[mn,mx],'r--'); axes[0].set_title('Actual vs Predicted')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')

residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=30, color='#a78bfa', edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--'); axes[1].set_title('Residuals Distribution')
plt.tight_layout()

## 8️⃣ Feature Importance (Coefficients)

In [ ]:
coefs = pd.Series(pipe.named_steps['model'].coef_, index=FEATURES).sort_values()
coefs.plot.barh(figsize=(10,5), color=['#f87171' if v<0 else '#34d399' for v in coefs.values])
plt.title('Linear Regression Coefficients'); plt.axvline(0, color='white', lw=0.8)
plt.tight_layout()

## Save the model

In [ ]:
import joblib
payload = {'pipeline': pipe, 'label_encoders': {'race_ethnicity': le}, 'features': FEATURES}
joblib.dump(payload, '../models/student_model.pkl')
print('Model saved!')